# Colab launcher — Optimize-Mars-Sl

Every member runs stages through `scripts/colab/run_stage.py`. Data and outputs live on Google Drive under `MyDrive/Text2SQL_Colab`.
See `COLAB.md`. Set `STAGE` and `RUN_ID`, run the cells in order. Rerunning with the same `RUN_ID` resumes from the newest checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BRANCH = 'feature/colab-runner'
import os, subprocess
if not os.path.exists('/content/Optimize-Mars-Sl'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/vietnguyen21/Optimize-Mars-Sl.git', '/content/Optimize-Mars-Sl'], check=True)
subprocess.run(['git', '-C', '/content/Optimize-Mars-Sl', 'pull', '-q', 'origin', BRANCH], check=True)
print(subprocess.run(['git', '-C', '/content/Optimize-Mars-Sl', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
# One-time per Drive: build data/*.jsonl and manifest.json from data/bird/{dev,train}.zip
!cd /content/Optimize-Mars-Sl && python scripts/colab/run_stage.py --stage prepare


In [ ]:
# Start a stage in the background so it survives the cell; log goes to /content/<RUN_ID>.log
STAGE = 'grounding'          # grounding | sft | grpo | system1 | evaluation
RUN_ID = 'm1-grounding-seed42'
EXTRA = ''                   # e.g. '--set MAX_STEPS=50 --set LAMBDA_JEPA=0'
subprocess.Popen(f'cd /content/Optimize-Mars-Sl && nohup python -u scripts/colab/run_stage.py --stage {STAGE} --run-id {RUN_ID} {EXTRA} > /content/{RUN_ID}.log 2>&1 &', shell=True)
print('started', STAGE, RUN_ID)


In [ ]:
# Status: rerun any time (also keeps the runtime active)
!date; grep -aE "STAGE_|Error|Traceback" /content/{RUN_ID}.log | tail -5; tail -c 300 /content/{RUN_ID}.log | tr '' '
' | tail -2; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader
